In [1]:
!pip install -U datasets

In [2]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
import torch

In [ ]:
!unzip mbart_lora_es_pt.zip -d mbart_lora_es_pt

In [3]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

In [4]:
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model     = M2M100ForConditionalGeneration.from_pretrained(model_name)
model = get_peft_model(model, peft_config)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
import huggingface_hub
huggingface_hub.login() # now you will be prompted to enter your token; enter it.

In [6]:
ds_es = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="dev")
ds_pt = load_dataset("openlanguagedata/flores_plus", "por_Latn", split="dev")
parallel_pt = [{"translation": {"es": e["text"], "pt": p["text"]}} for e, p in zip(ds_es, ds_pt)]

README.md:   0%|          | 0.00/72.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

spa_Latn.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

spa_Latn.parquet:   0%|          | 0.00/134k [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

por_Latn.parquet:   0%|          | 0.00/122k [00:00<?, ?B/s]

por_Latn.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [7]:
dataset_pt = Dataset.from_list(parallel_pt).train_test_split(test_size=0.1, seed=42)
train_pt = dataset_pt["train"]
eval_pt = dataset_pt["test"]

In [10]:
tokenizer.src_lang = "es"

def tokenize_es_pt(batch):
    # batch['translation'] es un dict {"es": ..., "pt": ...}
    src_texts = [ex["es"] for ex in batch["translation"]]
    tgt_texts = [ex["pt"] for ex in batch["translation"]]

    # ← idioma origen ya se indicó una sola vez con tokenizer.src_lang = "es"
    model_inputs = tokenizer(
        src_texts,
        max_length=128,
        padding="max_length",
        truncation=True
    )

    # Idioma destino solo se pone al tokenizar los labels
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            tgt_texts,
            max_length=128,
            padding="max_length",
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized_pt = train_pt.map(tokenize_es_pt, batched=True, remove_columns=train_pt.column_names)
eval_tokenized_pt  = eval_pt.map(tokenize_es_pt,  batched=True, remove_columns=eval_pt.column_names)

Map:   0%|          | 0/897 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [11]:
training_args_pt = Seq2SeqTrainingArguments(
    output_dir="./mbart_lora_es_pt",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-4,
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_pt",
    predict_with_generate=True,  # Importante para traducción
    fp16=torch.cuda.is_available(),  # Entrenamiento más rápido si tienes GPU
    save_total_limit=1,
    report_to="none",
    label_names=["labels"]
)

trainer_pt = Seq2SeqTrainer(
    model=model,
    args=training_args_pt,
    train_dataset=train_tokenized_pt,
    eval_dataset=eval_tokenized_pt,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

In [12]:
trainer_pt.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,No log,5.586637
2,No log,5.577768
3,5.735100,5.579812
4,5.735100,5.584342
5,5.521300,5.583287
6,5.521300,5.588004
7,5.487100,5.594456
8,5.487100,5.600443
9,5.449000,5.601642
10,5.449000,5.605652


TrainOutput(global_step=3375, training_loss=5.491941912615741, metrics={'train_runtime': 446.2331, 'train_samples_per_second': 30.152, 'train_steps_per_second': 7.563, 'total_flos': 3656988874506240.0, 'train_loss': 5.491941912615741, 'epoch': 15.0})

In [13]:
model.save_pretrained("./mbart_lora_es_pt")
tokenizer.save_pretrained("./mbart_lora_es_pt")
print("✅ Adaptadores LoRA para Español → Portugués guardados.")

✅ Adaptadores LoRA para Español → Portugués guardados.


In [29]:
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
base_model     = M2M100ForConditionalGeneration.from_pretrained(model_name)
model = PeftModel.from_pretrained(base_model, "./mbart_lora_es_pt")

In [30]:
model.train()
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 485,085,184 || trainable%: 0.2432


In [31]:
ds_gl = load_dataset("openlanguagedata/flores_plus", "glg_Latn", split="dev")
parallel_gl = [{"translation": {"es": e["text"], "gl": g["text"]}} for e, g in zip(ds_es, ds_gl)]

dataset_gl = Dataset.from_list(parallel_gl).train_test_split(test_size=0.1, seed=42)
train_gl = dataset_gl["train"]
eval_gl = dataset_gl["test"]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

In [33]:
# 1 · Idioma de origen (solo una vez)
tokenizer.src_lang = "es"        # español
tokenizer.tgt_lang = "gl"      # destino  ← ¡ESTO faltaba!

# 2 · Función de tokenización es → gl
def tokenize_es_gl(batch):
    src_texts = [ex["es"] for ex in batch["translation"]]
    tgt_texts = [ex["gl"] for ex in batch["translation"]]

    # Entrada
    model_inputs = tokenizer(
        src_texts,
        max_length=128,
        padding="max_length",
        truncation=True,
    )

    # Etiquetas (lengua destino)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            tgt_texts,
            max_length=128,
            padding="max_length",
            truncation=True,
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 3 · Mapeo sobre tus splits
train_tokenized_gl = train_gl.map(tokenize_es_gl, batched=True,
                                  remove_columns=train_gl.column_names)
eval_tokenized_gl  = eval_gl.map(tokenize_es_gl,  batched=True,
                                  remove_columns=eval_gl.column_names)

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [34]:
training_args_gl = Seq2SeqTrainingArguments(
    output_dir="./mbart_lora_es_gl",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-4,
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_gl",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none",
    label_names=["labels"]
)

trainer_gl = Seq2SeqTrainer(
    model=model,
    args=training_args_gl,
    train_dataset=train_tokenized_gl,
    eval_dataset=eval_tokenized_gl,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

In [35]:
trainer_gl.train()

Epoch,Training Loss,Validation Loss
1,No log,6.552587
2,No log,6.522985
3,No log,6.522641
4,No log,6.534904
5,No log,6.535249
6,No log,6.532786
7,No log,6.538885
8,No log,6.548510
9,No log,6.557878
10,No log,6.566730


('./mbart_lora_es_gl/tokenizer_config.json',
 './mbart_lora_es_gl/special_tokens_map.json',
 'mbart_lora_es_gl/vocab.json',
 'mbart_lora_es_gl/sentencepiece.bpe.model',
 './mbart_lora_es_gl/added_tokens.json')

In [43]:
model.save_pretrained("./m2m100_lora_es_gl")
tokenizer.save_pretrained("./m2m100_lora_es_gl")

('./m2m100_lora_es_gl/tokenizer_config.json',
 './m2m100_lora_es_gl/special_tokens_map.json',
 'm2m100_lora_es_gl/vocab.json',
 'm2m100_lora_es_gl/sentencepiece.bpe.model',
 './m2m100_lora_es_gl/added_tokens.json')

In [36]:
pip install evaluate sacrebleu

In [37]:
from evaluate import load

In [44]:
from typing import List

# ──────────────────────────────────────────────────────────────
#  Función de prueba en lote para es → gl
# ──────────────────────────────────────────────────────────────
def test_translation_batch(
    sentences_es: List[str],
    references_gl: List[str],
    model_path: str = "./m2m100_lora_es_gl",
):
    try:
        # 1 · Modelo base + adaptador LoRA
        base = "facebook/m2m100_418M"            # checkpoint multilingüe
        base_model = M2M100ForConditionalGeneration.from_pretrained(base)
        model      = PeftModel.from_pretrained(base_model, model_path)
        model.eval()

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)

        # 2 · Tokenizador
        tokenizer = M2M100Tokenizer.from_pretrained(base)   # o model_path si copiaste el vocabulario
        tokenizer.src_lang = "es"                           # idioma de entrada

        # 3 · Preparar lote de entradas
        inputs = tokenizer(
            sentences_es,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        ).to(device)

        # 4 · Generar traducciones (forzamos <gl>)
        gl_bos_id = tokenizer.get_lang_id("gl")
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=gl_bos_id,
                max_length=128,
                num_beams=4,
                early_stopping=True,
            )

        translations = tokenizer.batch_decode(generated, skip_special_tokens=True)

        # 5 · Mostrar pares
        for src, pred, ref in zip(sentences_es, translations, references_gl):
            print(f"ES: {src}")
            print(f"GL (pred): {pred}")
            print(f"GL (ref) : {ref}")
            print("-" * 60)

        # 6 · Métricas BLEU y chrF
        bleu = load("bleu")
        chrf = load("chrf")

        bleu_score = bleu.compute(
            predictions=translations,
            references=[[r] for r in references_gl],
        )
        chrf_score = chrf.compute(
            predictions=translations,
            references=references_gl,
        )

        print("\n📊 Métricas globales:")
        print(f"BLEU : {bleu_score['bleu']:.4f}")
        print(f"chrF : {chrf_score['score']:.2f}")

    except Exception as e:
        print(f"❌ Error en traducción: {e}")


In [45]:
ds_es = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="devtest[:50]")
ds_gl = load_dataset("openlanguagedata/flores_plus", "glg_Latn", split="devtest[:50]")

# Tomar una muestra de 50 ejemplos para visualización
sample_es = ds_es["text"][:50]
sample_gl = ds_gl["text"][:50]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

In [46]:
test_translation_batch(sample_es, sample_gl)

ES: «Actualmente, tenemos ratones de cuatro meses de edad que antes solían ser diabéticos y que ya no lo son», agregó.
GL (pred): O presidente da Cámara de Representantes da ONU, José Manuel Barroso, afirmou:
GL (ref) : "Agora temos ratos de 4 meses que xa non son diabéticos, pero que no seu momento si que o foron", engadiu.
------------------------------------------------------------
ES: La investigación todavía se ubica en su etapa inicial, conforme indicara el Dr. Ehud Ur, docente en la carrera de medicina de la Universidad de Dalhousie, en Halifax, Nueva Escocia, y director del departamento clínico y científico de la Asociación Canadiense de Diabetes.
GL (pred): A investigación aínda está na súa fase inicial, informou o profesor de medicina na Universidade de Dalhousie, en Halifax, Nova Escocia, e o director do departamento clínico e científico da Asociación Canadense de Diabetes.
GL (ref) : O Dr. Ehud Ur, profesor de medicina na Universidade Dalhousie en Halifax (Nova Escocia) e p


📊 Métricas globales:
BLEU : 0.0893
chrF : 32.20
